In [1]:
import pandas as pd

data = pd.read_csv('/Volumes/MoonDrive/ML/MIMICIII/data/NOTEEVENTS.csv')
data.head()

/var/folders/3f/zx0j8j0s7_s_cd9yv7xfmkz80000gn/T/ipykernel_20030/3241898916.py:3: DtypeWarning: Columns (0: CHARTTIME, 1: STORETIME) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/Volumes/MoonDrive/ML/MIMICIII/data/NOTEEVENTS.csv')


,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT
0,174,22532,167853.0,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...
1,175,13702,107527.0,2118-06-14,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-6-2**] Discharg...
2,176,13702,167118.0,2119-05-25,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2119-5-4**] D...
3,177,13702,196489.0,2124-08-18,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2124-7-21**] ...
4,178,26880,135453.0,2162-03-25,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2162-3-3**] D...


In [2]:
res = data[data['SUBJECT_ID'] == 10006]
res = res.sort_values(by='CHARTTIME', ascending=True)
res.head()

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT
844860,841673,10006,NaN,2164-09-25,2164-09-25 17:27:00,NaN,Radiology,CHEST (PORTABLE AP),NaN,NaN,[**2164-9-25**] 5:27 PM\n CHEST (PORTABLE AP) ...
832693,842142,10006,NaN,2164-09-29,2164-09-29 12:31:00,NaN,Radiology,LIVER OR GALLBLADDER US (SINGLE ORGAN),NaN,NaN,[**2164-9-29**] 12:31 PM\n LIVER OR GALLBLADDE...
833873,842211,10006,NaN,2164-09-30,2164-09-30 16:51:00,NaN,Radiology,CHEST (PORTABLE AP),NaN,NaN,[**2164-9-30**] 4:51 PM\n CHEST (PORTABLE AP) ...
843804,842219,10006,NaN,2164-09-30,2164-09-30 23:15:00,NaN,Radiology,CHEST (PORTABLE AP),NaN,NaN,[**2164-9-30**] 11:15 PM\n CHEST (PORTABLE AP)...
844307,842334,10006,NaN,2164-10-01,2164-10-01 16:48:00,NaN,Radiology,CT HEAD W/O CONTRAST,NaN,NaN,[**2164-10-1**] 4:48 PM\n CT HEAD W/O CONTRAST...


In [3]:
data['CATEGORY'].unique()

<ArrowStringArray>
['Discharge summary',              'Echo',               'ECG',
           'Nursing',        'Physician ',    'Rehab Services',
  'Case Management ',      'Respiratory ',         'Nutrition',
           'General',       'Social Work',          'Pharmacy',
           'Consult',         'Radiology',     'Nursing/other']
Length: 15, dtype: str

In [4]:
res = data[data['CATEGORY']== 'Discharge summary']
res.head()

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT
0,174,22532,167853.0,2151-08-04,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2151-7-16**] Dischar...
1,175,13702,107527.0,2118-06-14,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2118-6-2**] Discharg...
2,176,13702,167118.0,2119-05-25,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2119-5-4**] D...
3,177,13702,196489.0,2124-08-18,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2124-7-21**] ...
4,178,26880,135453.0,2162-03-25,NaN,NaN,Discharge summary,Report,NaN,NaN,Admission Date: [**2162-3-3**] D...


In [5]:
# Splitting the dataset

'''
Each discharge summary is a unique data point
there can be multiple discharge summaries for a single patient, but we will treat each discharge summary as a separate data point.
input to summarizsation for a discharge summary will be all the notes for that patient up to the time of the discharge summary (before previous discharge summary), and the output will be the discharge summary itself.

NOTE:  The deidentification process for structured data required the removal of all eighteen of the identifying data elements listed in HIPAA, including fields such as patient name, telephone number, address, and dates. In particular, dates were shifted into the future by a random offset for each individual patient in a consistent manner to preserve intervals, resulting in stays which occur sometime between the years 2100 and 2200
https://physionet.org/content/mimiciii/1.4/

'''

data['CHARTDATE'] = pd.to_datetime(data['CHARTDATE'], errors='coerce')

# drop rows with missing CHARTDATE
data = data[~data['CHARTDATE'].isna()]

discharge_summaries = data[data['CATEGORY'] == 'Discharge summary']
supporting_notes = data[data['CATEGORY'] != 'Discharge summary']

print(discharge_summaries.shape)
print(supporting_notes.shape)
print(len(discharge_summaries['SUBJECT_ID'].unique()))

(59652, 11)
(2023528, 11)
41127


In [6]:
from tqdm import tqdm

tqdm.pandas()

discharge_summaries = discharge_summaries.reset_index(drop=True)
discharge_summaries['SAMPLE_ID'] = discharge_summaries.index

def apply_sample_id_to_supporting_notes(row, discharge_summaries):
    '''
    Each discharge summary is a unique data point
    there can be multiple discharge summaries for a single patient, but we will treat each discharge summary as a separate data point.
    input to summarizsation for a discharge summary will be all the notes for that patient up to the time of the discharge summary (before previous discharge summary), and the output will be the discharge summary itself.
    '''
    subject_id = row['SUBJECT_ID']
    chart_time = row['CHARTDATE']


    # Find the discharge summary for the same subject_id and chart_time is greater than the chart_time of the supporting note
    discharge_summary = discharge_summaries[(discharge_summaries['SUBJECT_ID'] == subject_id) & (discharge_summaries['CHARTDATE'] > chart_time)]
    
    if discharge_summary.empty:
        return None
    else:
        # Get the discharge summary that is closest to the current supporting note(the one with the minimum chart_time)
        latest_discharge_summary = discharge_summary.loc[discharge_summary['CHARTDATE'].idxmin()]
        return latest_discharge_summary['SAMPLE_ID']

supporting_notes['SAMPLE_ID'] = supporting_notes.progress_apply(lambda row: apply_sample_id_to_supporting_notes(row, discharge_summaries), axis=1)


100%|██████████| 2023528/2023528 [10:45<00:00, 3134.76it/s]


In [7]:
# drop supporting notes that do not have a corresponding discharge summary
print(supporting_notes.shape)
supporting_notes = supporting_notes[~supporting_notes['SAMPLE_ID'].isna()]
print(supporting_notes.shape)

# dropping discharge summaries that do not have any supporting notes
print(discharge_summaries.shape)
discharge_summaries = discharge_summaries[discharge_summaries['SAMPLE_ID'].isin(supporting_notes['SAMPLE_ID'].unique())]
print(discharge_summaries.shape)

(2023528, 12)
(1839080, 12)
(59652, 12)
(54179, 12)


In [9]:
import pandas as pd 

discharge_summaries.to_json('/Volumes/MoonSSD/ML/discharge_notes_summarizer/discharge_summaries.json')
supporting_notes.to_json('/Volumes/MoonSSD/ML/discharge_notes_summarizer/supporting_notes.json')

discharge_summaries = pd.read_json('/Volumes/MoonSSD/ML/discharge_notes_summarizer/discharge_summaries.json')
supporting_notes = pd.read_json('/Volumes/MoonSSD/ML/discharge_notes_summarizer/supporting_notes.json')

/var/folders/3f/zx0j8j0s7_s_cd9yv7xfmkz80000gn/T/ipykernel_20030/3504604170.py:3: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  discharge_summaries.to_json('/Volumes/MoonSSD/ML/discharge_notes_summarizer/discharge_summaries.json')
/var/folders/3f/zx0j8j0s7_s_cd9yv7xfmkz80000gn/T/ipykernel_20030/3504604170.py:4: Pandas4Warning: The default 'epoch' date format is deprecated and will be removed in a future version, please use 'iso' date format instead.
  supporting_notes.to_json('/Volumes/MoonSSD/ML/discharge_notes_summarizer/supporting_notes.json')


In [10]:
supporting_notes.head()

,ROW_ID,SUBJECT_ID,HADM_ID,CHARTDATE,CHARTTIME,STORETIME,CATEGORY,DESCRIPTION,CGID,ISERROR,TEXT,SAMPLE_ID
52108,59653,31038,174978.0,4179686400000,NaN,NaN,Echo,Report,NaN,NaN,PATIENT/TEST INFORMATION:\nIndication: Endocar...,125
52109,59654,70150,156140.0,6985267200000,NaN,NaN,Echo,Report,NaN,NaN,"PATIENT/TEST INFORMATION:\nIndication: CHF, A-...",129
52110,59655,54190,188571.0,5936716800000,NaN,NaN,Echo,Report,NaN,NaN,PATIENT/TEST INFORMATION:\nIndication: Cerebro...,130
52111,59656,5771,185291.0,6427209600000,NaN,NaN,Echo,Report,NaN,NaN,PATIENT/TEST INFORMATION:\nIndication: 78 year...,136
52112,59657,80030,100442.0,4715539200000,NaN,NaN,Echo,Report,NaN,NaN,PATIENT/TEST INFORMATION:\nIndication: Left ve...,138


In [11]:
print(discharge_summaries.shape)
print(supporting_notes.shape)

# only keep discharge summaries that have supporting notes
discharge_summaries = discharge_summaries[discharge_summaries['SAMPLE_ID'].isin(supporting_notes['SAMPLE_ID'].unique())]
print(discharge_summaries.shape)
print(supporting_notes.shape)

(54179, 12)
(1839080, 12)
(54179, 12)
(1839080, 12)


In [ ]:
from sklearn.model_selection import train_test_split

# split sample IDs in to val and test
# Make sure patient IDs are not shared between train, val and test sets

subject_ids = discharge_summaries['SUBJECT_ID'].unique()
print(len(subject_ids))

train_subject_ids, test_subject_ids = train_test_split(subject_ids, test_size=0.05, random_state=42)
train_subject_ids, val_subject_ids = train_test_split(train_subject_ids, test_size=0.05, random_state=42)

supporting_notes['SPLIT'] = supporting_notes['SUBJECT_ID'].apply(lambda x: 'train' if x in train_subject_ids else ('val' if x in val_subject_ids else 'test'))
discharge_summaries['SPLIT'] = discharge_summaries['SUBJECT_ID'].apply(lambda x: 'train' if x in train_subject_ids else ('val' if x in val_subject_ids else 'test'))    

train_inputs = supporting_notes[supporting_notes['SPLIT'] == 'train']
train_outputs = discharge_summaries[discharge_summaries['SPLIT'] == 'train']
val_inputs = supporting_notes[supporting_notes['SPLIT'] == 'val']
val_outputs = discharge_summaries[discharge_summaries['SPLIT'] == 'val']
test_inputs = supporting_notes[supporting_notes['SPLIT'] == 'test']
test_outputs = discharge_summaries[discharge_summaries['SPLIT'] == 'test']
print(train_inputs.shape, train_outputs.shape)
print(val_inputs.shape, val_outputs.shape)
print(test_inputs.shape, test_outputs.shape)

# assert no leakage of sample ID and subject ID across train, val and test sets
assert set(train_inputs['SAMPLE_ID'].unique()).isdisjoint(set(val_inputs['SAMPLE_ID'].unique()))
assert set(train_inputs['SAMPLE_ID'].unique()).isdisjoint(set(test_inputs['SAMPLE_ID'].unique()))
assert set(val_inputs['SAMPLE_ID'].unique()).isdisjoint(set(test_inputs['SAMPLE_ID'].unique())) 
assert set(train_inputs['SUBJECT_ID'].unique()).isdisjoint(set(val_inputs['SUBJECT_ID'].unique()))
assert set(train_inputs['SUBJECT_ID'].unique()).isdisjoint(set(test_inputs['SUBJECT_ID'].unique()))
assert set(val_inputs['SUBJECT_ID'].unique()).isdisjoint(set(test_inputs['SUBJECT_ID'].unique()))       

# assert that for each discharge summary in the train, val and test sets, there are supporting notes with the same sample ID in the corresponding train, val and test sets
for sample_id in train_outputs['SAMPLE_ID'].unique():
    assert sample_id in train_inputs['SAMPLE_ID'].unique()
for sample_id in val_outputs['SAMPLE_ID'].unique():
    assert sample_id in val_inputs['SAMPLE_ID'].unique()
for sample_id in test_outputs['SAMPLE_ID'].unique():
    assert sample_id in test_inputs['SAMPLE_ID'].unique()


40510


In [ ]:
#test validitity of sample ID assignment


In [ ]:
train_inputs.to_json('/Volumes/MoonSSD/ML/discharge_notes_summarizer/train_inputs.json')
train_outputs.to_json('/Volumes/MoonSSD/ML/discharge_notes_summarizer/train_outputs.json')
val_inputs.to_json('/Volumes/MoonSSD/ML/discharge_notes_summarizer/val_inputs.json')
val_outputs.to_json('/Volumes/MoonSSD/ML/discharge_notes_summarizer/val_outputs.json')
test_inputs.to_json('/Volumes/MoonSSD/ML/discharge_notes_summarizer/test_inputs.json')
test_outputs.to_json('/Volumes/MoonSSD/ML/discharge_notes_summarizer/test_outputs.json')


OSError: Cannot save file into a non-existent directory: '/Volumes/MoonDrive/ML/MIMICIII/data/discharge_notes_summarizer'